In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Files (250).RTF to Files (250).RTF


In [ ]:
!pip install striprtf

In [ ]:
from striprtf.striprtf import rtf_to_text

with open("Files (250).RTF", "r", encoding="utf-8", errors="ignore") as f:
    rtf = f.read()

text = rtf_to_text(rtf)

In [ ]:
articles = text.split("End of Document")

In [ ]:
import re
import pandas as pd

data = []

for article in articles:

    # дата
    date_match = re.search(
        r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}',
        article
    )

    if not date_match:
        continue

    date = pd.to_datetime(date_match.group())

    body_match = re.search(r'Body(.*)', article, re.DOTALL)

    if body_match:
        body = body_match.group(1)

        body = re.sub(r'Load-Date:.*', '', body, flags=re.DOTALL)
        body = re.sub(r'\s+', ' ', body)

        data.append({
            'date': date,
            'text': body
        })

df = pd.DataFrame(data)

In [ ]:
len(monthly_news)

3

In [ ]:
df['month'] = df['date'].dt.to_period('M')

monthly_news = (
    df.groupby('month')['text']
    .apply(lambda x: ' '.join(x))
    .reset_index()
)

monthly_news['month'] = monthly_news['month'].dt.to_timestamp()

In [ ]:
df.tail()

,date,text,next_ifo_date,target,business_climate,month
245,2026-05-16,UK inflation is set to have eased last month ...,NaT,NaN,NaN,2026-05
246,2026-05-16,Paddleboarders can now skip the hard work of ...,NaT,NaN,NaN,2026-05
247,2026-05-16,A selloff in global bonds halted a rally in s...,NaT,NaN,NaN,2026-05
248,2026-05-16,UK INFLATION is set to have eased last month ...,NaT,NaN,NaN,2026-05
249,2026-05-16,Paddleboarders can now skip the hard work of ...,NaT,NaN,NaN,2026-05


In [ ]:
monthly_news

,month,text
0,2026-03-01,Educators continue to debate a question that ...
1,2026-04-01,President Javier Milei weighed in again on th...
2,2026-05-01,Mumbai: Hindustan Unilever (HUL) has hiked pr...


## IFO data

In [ ]:
uploaded_ifo  = files.upload()

Saving gsk-e-202604.xlsx to gsk-e-202604.xlsx


In [ ]:
ifo_filename = list(uploaded_ifo.keys())[0]

print(ifo_filename)

gsk-e-202604.xlsx


In [ ]:
ifo = pd.read_excel(
    ifo_filename,
    sheet_name='ifo Business Climate',
    skiprows=9,
    usecols=[0,1,2,3]
)

ifo.columns = [
    'month',
    'business_climate',
    'business_situation',
    'business_expectations'
]

ifo.head()

,month,business_climate,business_situation,business_expectations
0,02/2005,92.0,88.0,96.2
1,03/2005,90.1,85.8,94.5
2,04/2005,89.9,86.3,93.7
3,05/2005,89.3,86.1,92.7
4,06/2005,89.3,85.6,93.2


In [ ]:
ifo['month'] = pd.to_datetime(
    ifo['month'].astype(str).str.strip(),
    format='%m/%Y'
)

ifo = ifo.dropna(subset=['business_climate'])

ifo['future_change'] = (
    ifo['business_climate'].shift(-1)
    - ifo['business_climate']
)

ifo['target'] = (ifo['future_change'] > 0).astype(int)

ifo.head()

,month,business_climate,business_situation,business_expectations,future_change,target
0,2005-02-01,92.0,88.0,96.2,-1.9,0
1,2005-03-01,90.1,85.8,94.5,-0.2,0
2,2005-04-01,89.9,86.3,93.7,-0.6,0
3,2005-05-01,89.3,86.1,92.7,0.0,0
4,2005-06-01,89.3,85.6,93.2,1.8,1


## If we want to do as a Time Series

In [ ]:
dataset = monthly_news.merge(
    ifo[['month', 'business_climate', 'target']],
    on='month',
    how='inner'
)

In [ ]:
dataset.head(19)

,month,text,business_climate,target
0,2026-03-01,Educators continue to debate a question that ...,86.3,0
1,2026-04-01,By Adrian Peel adrian.peel@iliffemedia.co.uk ...,84.4,0


## Else

In [ ]:
ifo = ifo.sort_values('month')
df = df.sort_values('date')

In [ ]:
ifo

,month,business_climate,business_situation,business_expectations,future_change,target
0,2005-02-01,92.0,88.0,96.2,-1.9,0
1,2005-03-01,90.1,85.8,94.5,-0.2,0
2,2005-04-01,89.9,86.3,93.7,-0.6,0
3,2005-05-01,89.3,86.1,92.7,0.0,0
4,2005-06-01,89.3,85.6,93.2,1.8,1
...,...,...,...,...,...,...
250,2025-12-01,87.7,85.6,89.8,0.0,0
251,2026-01-01,87.7,85.6,89.7,0.8,1
252,2026-02-01,88.5,86.7,90.3,-2.2,0
253,2026-03-01,86.3,86.7,85.9,-1.9,0


In [ ]:
df

,date,text
45,2026-03-31,Educators continue to debate a question that ...
146,2026-04-17,President Javier Milei weighed in again on th...
229,2026-04-27,"During the first half of April, annual inflat..."
203,2026-04-29,The Economic and Financial Crimes Commission ...
8,2026-04-29,By Adrian Peel adrian.peel@iliffemedia.co.uk ...
...,...,...
54,2026-05-16,These are undoubtedly challenging times; low ...
232,2026-05-16,CIARA O BRIEN Global equity indexes fell yest...
75,2026-05-16,Inflation in the UK is expected to temporaril...
118,2026-05-16,A selloff in global bonds halted a rally in s...


In [ ]:
import pandas as pd
import numpy as np

df = df.sort_values('date').reset_index(drop=True)
ifo = ifo.sort_values('month').reset_index(drop=True)

ifo = ifo.dropna(subset=['future_change'])

df['next_ifo_date'] = pd.NaT
df['target'] = np.nan
df['business_climate'] = np.nan

for i, row in df.iterrows():

    article_date = row['date']

    matched = False

    for _, ifo_row in ifo.iterrows():

        release_date = ifo_row['month']
        cutoff_date = (
            release_date - pd.DateOffset(months=1)
        ).replace(day=15)

        if article_date <= cutoff_date:

            df.loc[i, 'next_ifo_date'] = release_date
            df.loc[i, 'target'] = ifo_row['target']
            df.loc[i, 'business_climate'] = ifo_row['business_climate']

            matched = True
            break

In [ ]:
df

,date,text,next_ifo_date,target,business_climate
0,2026-03-31,Educators continue to debate a question that ...,NaT,NaN,NaN
1,2026-04-17,President Javier Milei weighed in again on th...,NaT,NaN,NaN
2,2026-04-27,"During the first half of April, annual inflat...",NaT,NaN,NaN
3,2026-04-29,The Economic and Financial Crimes Commission ...,NaT,NaN,NaN
4,2026-04-29,By Adrian Peel adrian.peel@iliffemedia.co.uk ...,NaT,NaN,NaN
...,...,...,...,...,...
245,2026-05-16,UK inflation is set to have eased last month ...,NaT,NaN,NaN
246,2026-05-16,Paddleboarders can now skip the hard work of ...,NaT,NaN,NaN
247,2026-05-16,A selloff in global bonds halted a rally in s...,NaT,NaN,NaN
248,2026-05-16,UK INFLATION is set to have eased last month ...,NaT,NaN,NaN
